# <center>Motivação</center>

<p>Garantir que o pré-treinamento do vgg19 possa ocorrer considerando um <code>UpSampling2D</code>. A máquina local não pode suportar tanta conta assim, o que torna impossível o treinamento.</p>

## 1) Importar bibliotecas

In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2B2, preprocess_input
# import VGG19, preprocess_input
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np


In [ ]:
tf.__version__

## 2) Importar base de dados

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
classes = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck",
}

## 3) Definir funções de treino e teste

### 3.1) <code>early_stopping</code>

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    min_delta = 1E-4,
    patience = 5,
    verbose = 1,
    monitor = "val_loss",
)

### 3.2) <code>model_compile</code>

In [ ]:
model_compile = {
    "optimizer": tf.keras.optimizers.Adam(1E-4),
    "loss": tf.keras.losses.SparseCategoricalCrossentropy(),
    "metrics": tf.keras.metrics.SparseCategoricalAccuracy()
}

## 4) Treinando modelo

### 4.1) Invocando base_model

<p>Vamos importar um modelo pré-treinado <code>vgg19</code>, capaz de interpretar melhor imagens referentes aos <code>ciffar10</code> com facilidade.</p>

In [ ]:
base_model = EfficientNetV2B2(
  include_top = False,
  weights = 'imagenet',
  # Segundo documentação, EfficientNetV2B2 foi treinado
  # com imagens de shape (260, 260, 3):
  input_shape = (260, 260, 3),
)

base_model.trainable = False

base_model.summary()

### 4.2) Definindo modelo

In [ ]:
model = tf.keras.Sequential([

    tf.keras.layers.InputLayer(
        shape = (32, 32, 3)
    ),

    # Como temos imagens com formatos (32, 32, 3), precisamos
    # garantir que o modelo consiga passar para o base_model (EfficientNetV2B2)
    # as mesmas dimensões em que foi treinado (MUITO PESADO):
    tf.keras.layers.Resizing(height = 260, width = 260, interpolation = 'bilinear'),

    base_model,
    tf.keras.layers.GlobalMaxPooling2D(),

    tf.keras.layers.Dense(4096, activation = "relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(4096, activation = "relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense( units = 10, activation = "softmax", )
])

model.compile(
    optimizer = model_compile["optimizer"],
    loss = model_compile["loss"],
    metrics = [model_compile["metrics"]],
)

model.summary(show_trainable = True)

### 3.3) Fit do modelo

In [ ]:
history = model.fit(
    x = preprocess_input(X_train),
    y = y_train,
    batch_size = 256,
    epochs = 30,
    validation_split = 0.2,
    callbacks = [early_stopping]
)

In [ ]:
# Conferindo performance do primeiro treinamento:
model.evaluate(
    preprocess_input(X_test), y_test
)

### 3.4) Qualidade do modelo

In [ ]:
fig, ax = plt.subplots()

ax.plot(
  history.epoch,
  history.history['loss']
)
ax.plot(
    history.epoch,
    history.history["val_loss"]
)

plt.title("Evolução do modelo conforme treinamento")
plt.show()

### 3.5) Fining tunning

<p>Apesar de parecer que treinamos o modelo duas vezes (ou melhor, as camadas dense duas vezes - o que foi), é necessário o treinamento inicial dessas camadas porque, caso contrário, o feedback que elas passariam para todo o corpo da CNN seria tão ruim nas primeiras etapas de aprendizado que todo o <code>EfficientNetV2B2</code> iria perder sua precisão em detrimento das falhas das últimas camadas.

Assim posto, precisamento agora retreinar o modelo - <i>fine tunning</i> - de forma que a incipiência das camadas dense não prejudique tanto o modelo.</p>

#### 3.5.1) Reconfigurar <code>base_model</code>

In [ ]:
# Portanto, redefinir base_model.trainable para treinamento:
base_model.trainable = True
model.summary(show_trainable = True)

#### 3.5.2) Treinar novamente

<p>Aqui, treinamos novamente considerando as novas camadas dense treinadas já anteriormente com <code>base_model.treinable = False</code>.</p>

In [ ]:
history_v2 = model.fit(
    x = preprocess_input(X_train),
    y = y_train,
    epochs = 50,
    batch_size = 128,
    validation_split = 0.2,
    callbacks = [early_stopping]
)

#### 3.5.4) Verificar qualidade do treinamento.

In [ ]:
fig, ax = plt.subplots()

ax.plot(
  history_v2.epoch,
  history_v2.history['loss']
)
ax.plot(
    history_v2.epoch,
    history_v2.history["val_loss"]
)

plt.title("Evolução do modeloV2 conforme treinamento")
plt.show()

#### 3.5.5) Evaluate

In [ ]:
model.evaluate(
    preprocess_input(X_test),
    y_test
)

### 3.6) Salvar modelo

In [ ]:
model.save("./drive/MyDrive/models/cifar10.keras")